# E-007 — Red desde cero: ECG de 10 s frente al formato del PDF (EchoNext)

**Para qué sirve:** entrena **la misma red** dos veces sobre EchoNext: con los 10 s completos de las 12 derivaciones y con solo lo que trae el PDF del instituto (12 × 2,5 s + tira de 10 s de V1), usando el formato único con máscara. Mide dos cosas: (1) si la red supera la vara de las referencias (boosting tabular de E-005, AUC 0,695) y (2) **cuánto se pierde por el formato del PDF**, un resultado que no está publicado. Registro en Obsidian: *Experimentos — Modelo IA* → E-007; *Decisiones* → D-004, D-007, D-011, D-012.

**Datos:** `MyDrive/modelo-hvi/datos/echonext/` (señales, metadata). Licencia restringida de PhysioNet: no compartir la carpeta ni copiarla al vault. Los pesos entrenados quedan en `MyDrive/modelo-hvi/modelos/` (privado, derivan de datos restringidos).

**Salida:** solo agregados → `MyDrive/modelo-hvi/resultados/E-007_red_formato_pdf.json` y `.png`.

**Reglas:**
- Nunca imprimir ni graficar el ECG de un paciente individual.
- val elige la época y el umbral; test se mira **una sola vez** al final.
- Antes de compartir: *Editar → Borrar todos los resultados*.
- El código se versiona en `github.com/luchoaquinopla/modelo-hvi` (`notebooks/` y el paquete `modelo_hvi`).

**Cómo correrlo:** *Entorno de ejecución → Cambiar tipo de entorno → GPU T4*, después *Ejecutar todas*. Necesita el paquete `modelo_hvi` copiado del repo en `MyDrive/modelo-hvi/codigo/`, con el commit de origen en `codigo/COMMIT`. Tarda ~1–1,5 h; si Colab se desconecta, volver a ejecutar: las redes ya entrenadas se cargan desde Drive.

In [ ]:
# Celda 1 — Entorno, paquete del formato único y rutas (datos restringidos: solo Drive privado y disco local de Colab)
import json
import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from google.colab import drive

assert torch.cuda.is_available(), 'activar GPU: Entorno de ejecución → Cambiar tipo de entorno → GPU T4'
drive.mount('/content/drive')
CARPETA = Path('/content/drive/MyDrive/modelo-hvi')
DATOS = CARPETA / 'datos' / 'echonext'
LOCAL = Path('/content/e007')    # disco local: leer lotes desde Drive es lento
MODELOS = CARPETA / 'modelos'    # pesos entrenados con datos restringidos: solo Drive privado
SEMILLA = 20260912
ETIQUETA = 'lvwt_gte_13_flag'

# Paquete del formato único copiado del repo a Drive (el repo es privado); COMMIT registra de qué versión salió
CODIGO = CARPETA / 'codigo'
sys.path.insert(0, str(CODIGO))
VERSION_PAQUETE = (CODIGO / 'COMMIT').read_text().strip()
from modelo_hvi.formato_unico import MODO_COMPLETO, MODO_PDF, armar_entrada, mascara_esquema_pdf
from modelo_hvi.adaptadores.echonext import AdaptadorEchoNext

GPU = torch.cuda.get_device_name(0)
print('GPU:', GPU, '· modelo_hvi del commit', VERSION_PAQUETE)

In [ ]:
# Celda 2 — EchoNext al formato único (D-004/D-007): filtro de 40 Hz y marca de tramos vacíos, una vez por partición
# Se guarda el modo completo; el esquema del PDF se aplica al armar cada lote porque la señal filtrada es la misma
metadata = pd.read_csv(DATOS / 'echonext_metadata_100k.csv')
PARTICIONES = ['train', 'val', 'test']
adaptador = AdaptadorEchoNext()
formato = {}
for p in PARTICIONES:
    salida = LOCAL / p
    agregados = salida / 'agregados.json'
    if not agregados.exists():
        inicio = time.time()
        resumen = adaptador.convertir_archivo(DATOS / f'EchoNext_{p}_waveforms.npy', salida, MODO_COMPLETO)
        resumen['minutos'] = round((time.time() - inicio) / 60, 1)
        agregados.write_text(json.dumps(resumen))
    formato[p] = json.loads(agregados.read_text())
    print(p, formato[p])

senal = {p: np.load(LOCAL / p / 'senal.npy', mmap_mode='r') for p in PARTICIONES}
mascara = {p: np.load(LOCAL / p / 'mascara.npy', mmap_mode='r') for p in PARTICIONES}
filas = {p: metadata[metadata.split == p].reset_index(drop=True) for p in PARTICIONES}

# Control: señales y metadata en el mismo orden de filas (mismo chequeo que E-006, ahora en las tres particiones)
alineacion = {}
for p in PARTICIONES:
    assert len(filas[p]) == len(senal[p]), f'{p}: metadata y señales con distinta cantidad de filas'
    tab = np.load(DATOS / f'EchoNext_{p}_tabular_features.npy')
    alineacion[p] = round(float(max(abs(np.corrcoef(tab[:, j], filas[p].age_at_ecg)[0, 1]) for j in range(tab.shape[1]))), 4)
    assert alineacion[p] > 0.99, f'{p}: la edad no coincide con las features tabulares'
print('correlación de la edad con las features tabulares:', alineacion)

In [ ]:
# Celda 3 — Datos de la cabecera (las 6 variables de E-005, todas están en el PDF) y pesos por paciente
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

COLUMNAS = ['age_at_ecg', 'sexo_masculino', 'ventricular_rate', 'pr_interval', 'qrs_duration', 'qt_corrected']
CONTINUAS = [c for c in COLUMNAS if c != 'sexo_masculino']
for p in PARTICIONES:
    filas[p]['sexo_masculino'] = filas[p].sex.map({'male': 1, 'female': 0})
    assert filas[p].sexo_masculino.notna().all(), 'sexo con valores inesperados'

preparador = ColumnTransformer(
    [('continuas', make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler()), CONTINUAS)],
    remainder='passthrough',
).fit(filas['train'][COLUMNAS])
tabular = {p: preparador.transform(filas[p][COLUMNAS]).astype(np.float32) for p in PARTICIONES}
y = {p: filas[p][ETIQUETA].to_numpy().astype(np.float32) for p in PARTICIONES}

# Cada paciente pesa lo mismo en train (hay pacientes con muchos ECG, E-003)
peso_por_paciente = 1 / filas['train'].groupby('patient_key').patient_key.transform('size').to_numpy()
peso_red = (peso_por_paciente / peso_por_paciente.mean()).astype(np.float32)  # media 1: no cambia la escala del aprendizaje
print({p: tabular[p].shape for p in PARTICIONES}, 'prevalencia:', {p: round(float(y[p].mean()), 3) for p in PARTICIONES})

In [ ]:
# Celda 4 — Lotes en la GPU y red convolucional 1D (ECG con máscara + cabecera)
import torch.nn as nn

DISPOSITIVO = torch.device('cuda')
ESQUEMA_PDF = torch.from_numpy(mascara_esquema_pdf()).to(DISPOSITIVO)


def armar_lote(p, indices, modo):
    indices = np.sort(indices)  # lectura en orden del disco
    x = torch.from_numpy(np.asarray(senal[p][indices], dtype=np.float32)).to(DISPOSITIVO)
    m = torch.from_numpy(np.asarray(mascara[p][indices])).to(DISPOSITIVO)
    if modo == MODO_PDF:
        m = m & ESQUEMA_PDF
    entrada = torch.cat([x * m, m.float()], dim=1)  # lo mismo que armar_entrada del paquete, vectorizado
    return entrada, torch.from_numpy(tabular[p][indices]).to(DISPOSITIVO), torch.from_numpy(y[p][indices]).to(DISPOSITIVO), indices


# Control: el lote de la GPU coincide con armar_entrada del paquete
e, _, _, idx = armar_lote('val', np.arange(8), MODO_PDF)
referencia = np.stack([armar_entrada(np.asarray(senal['val'][i], np.float32), np.asarray(mascara['val'][i]) & mascara_esquema_pdf()) for i in idx])
assert np.allclose(e.cpu().numpy(), referencia, atol=1e-6), 'el lote de la GPU no coincide con el formato único'


class BloqueResidual(nn.Module):
    def __init__(self, entrada, salida, paso):
        super().__init__()
        self.camino = nn.Sequential(
            nn.Conv1d(entrada, salida, 7, paso, 3, bias=False), nn.BatchNorm1d(salida), nn.ReLU(),
            nn.Conv1d(salida, salida, 7, 1, 3, bias=False), nn.BatchNorm1d(salida))
        self.atajo = (nn.Identity() if entrada == salida and paso == 1
                      else nn.Sequential(nn.Conv1d(entrada, salida, 1, paso, bias=False), nn.BatchNorm1d(salida)))

    def forward(self, x):
        return torch.relu(self.camino(x) + self.atajo(x))


class RedHvi(nn.Module):
    def __init__(self, n_tabular, canales=24, anchos=(32, 64, 128, 256)):
        super().__init__()
        capas = [nn.Conv1d(canales, anchos[0], 15, 2, 7, bias=False), nn.BatchNorm1d(anchos[0]), nn.ReLU()]
        previo = anchos[0]
        for i, ancho in enumerate(anchos):
            capas += [BloqueResidual(previo, ancho, 1 if i == 0 else 2), BloqueResidual(ancho, ancho, 1)]
            previo = ancho
        self.ecg = nn.Sequential(*capas, nn.AdaptiveAvgPool1d(1), nn.Flatten())
        self.cabecera = nn.Sequential(nn.Linear(n_tabular, 32), nn.ReLU())
        self.salida = nn.Sequential(nn.Dropout(0.3), nn.Linear(previo + 32, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, 1))

    def forward(self, ecg, cabecera):
        return self.salida(torch.cat([self.ecg(ecg), self.cabecera(cabecera)], dim=1)).squeeze(1)


PARAMETROS = sum(t.numel() for t in RedHvi(tabular['train'].shape[1]).parameters())
print('parámetros de la red:', PARAMETROS)

In [ ]:
# Celda 5 — Entrenamiento: misma red, misma semilla y mismas reglas en los dos modos; val elige la época
from sklearn.metrics import roc_auc_score

EPOCAS, PACIENCIA, LOTE, TASA = 30, 5, 256, 1e-3
MODELOS.mkdir(parents=True, exist_ok=True)
peso_gpu = torch.from_numpy(peso_red).to(DISPOSITIVO)


def predecir(red, p, modo):
    red.eval()
    salida = np.empty(len(y[p]), dtype=np.float32)
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
        for i in range(0, len(y[p]), 1024):
            e, t, _, idx = armar_lote(p, np.arange(i, min(i + 1024, len(y[p]))), modo)
            salida[idx] = torch.sigmoid(red(e, t).float()).cpu().numpy()
    return salida


def entrenar(modo):
    ruta = MODELOS / f'E-007_{modo}.pt'
    torch.manual_seed(SEMILLA)
    rng = np.random.default_rng(SEMILLA)
    red = RedHvi(tabular['train'].shape[1]).to(DISPOSITIVO)
    if ruta.exists():  # reanudable si Colab se desconecta
        guardado = torch.load(ruta, map_location=DISPOSITIVO)
        red.load_state_dict(guardado['pesos'])
        return red, guardado['historial']
    optimizador = torch.optim.AdamW(red.parameters(), lr=TASA, weight_decay=1e-2)
    pasos = int(np.ceil(len(y['train']) / LOTE))
    programa = torch.optim.lr_scheduler.OneCycleLR(optimizador, max_lr=TASA, total_steps=EPOCAS * pasos)
    escalador = torch.amp.GradScaler()
    perdida_por_ecg = nn.BCEWithLogitsLoss(reduction='none')
    historial, mejor_auc, mejores_pesos, sin_mejora = [], -1.0, None, 0
    for epoca in range(EPOCAS):
        red.train()
        inicio, total = time.time(), 0.0
        orden = rng.permutation(len(y['train']))
        for i in range(0, len(orden), LOTE):
            e, t, objetivo, idx = armar_lote('train', orden[i:i + LOTE], modo)
            with torch.autocast('cuda', dtype=torch.float16):
                logits = red(e, t)
            perdida = (perdida_por_ecg(logits.float(), objetivo) * peso_gpu[torch.from_numpy(idx).to(DISPOSITIVO)]).mean()
            optimizador.zero_grad(set_to_none=True)
            escalador.scale(perdida).backward()
            escalador.step(optimizador)
            escalador.update()
            programa.step()
            total += perdida.item() * len(idx)
        auc_val = float(roc_auc_score(y['val'], predecir(red, 'val', modo)))
        historial.append({'epoca': epoca + 1, 'perdida_train': round(total / len(orden), 4),
                          'auc_val': round(auc_val, 4), 'segundos': round(time.time() - inicio)})
        print(modo, historial[-1])
        if auc_val > mejor_auc:
            mejor_auc, sin_mejora = auc_val, 0
            mejores_pesos = {k: v.detach().clone() for k, v in red.state_dict().items()}
        else:
            sin_mejora += 1
            if sin_mejora >= PACIENCIA:
                break
    red.load_state_dict(mejores_pesos)
    torch.save({'pesos': mejores_pesos, 'historial': historial}, ruta)
    return red, historial


MODOS = [MODO_COMPLETO, MODO_PDF]
historiales, predicciones = {}, {}
for modo in MODOS:
    red, historiales[modo] = entrenar(modo)
    predicciones[modo] = {p: predecir(red, p, modo) for p in ['val', 'test']}
    del red
    torch.cuda.empty_cache()

In [ ]:
# Celda 6 — Evaluación: umbral de Youden en val, test una sola vez, bootstrap pareado contra la vara (boosting de E-005)
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, brier_score_loss, roc_curve

TRAMOS_EDAD = [18, 40, 50, 60, 70, 80, 91]
prueba = filas['test'].copy()
prueba['tramo_edad'] = pd.cut(prueba.age_at_ecg, TRAMOS_EDAD, right=False).astype(str)
y_val, y_test = y['val'].astype(int), y['test'].astype(int)

# Vara: boosting con la configuración que ya eligió val en E-005 (acá no se elige nada)
boosting = HistGradientBoostingClassifier(learning_rate=0.05, max_leaf_nodes=15, max_iter=100, early_stopping=False, random_state=SEMILLA)
boosting.fit(filas['train'][COLUMNAS], filas['train'][ETIQUETA], sample_weight=peso_por_paciente)
predicciones['boosting_tabular'] = {p: boosting.predict_proba(filas[p][COLUMNAS])[:, 1] for p in ['val', 'test']}
NOMBRES = {MODO_COMPLETO: 'red_10s', MODO_PDF: 'red_formato_pdf', 'boosting_tabular': 'boosting_tabular'}
puntajes = {NOMBRES[k]: v['test'] for k, v in predicciones.items()}


def umbral_youden(yv, p):
    fpr, tpr, umbrales = roc_curve(yv, p)
    return float(umbrales[np.argmax(tpr - fpr)])


def auc_seguro(yv, p):
    # None si el subgrupo tiene una sola clase (ej. un tramo de edad sin casos con HVI)
    return round(float(roc_auc_score(yv, p)), 4) if 0 < yv.sum() < len(yv) else None


def metricas(yv, p, umbral):
    marcado = p >= umbral
    return {'n': int(len(yv)), 'positivos': int(yv.sum()), 'auc': auc_seguro(yv, p),
            'sensibilidad': round(float(marcado[yv == 1].mean()), 4) if yv.sum() else None,
            'especificidad': round(float((~marcado)[yv == 0].mean()), 4) if (yv == 0).any() else None}


def bootstrap_auc(yv, puntajes, repeticiones=1000):
    # Remuestreo pareado: los mismos pacientes para todos los modelos (igual que E-005 y E-006)
    rng = np.random.default_rng(SEMILLA)
    valores = {n: [] for n in puntajes}
    for _ in range(repeticiones):
        i = rng.integers(0, len(yv), len(yv))
        if 0 < yv[i].sum() < len(yv):
            for n, p in puntajes.items():
                valores[n].append(roc_auc_score(yv[i], p[i]))
    return {n: np.array(v) for n, v in valores.items()}


def intervalo(v):
    return [round(float(np.percentile(v, 2.5)), 4), round(float(np.percentile(v, 97.5)), 4)]


muestras = bootstrap_auc(y_test, puntajes)
umbrales = {NOMBRES[k]: umbral_youden(y_val, v['val']) for k, v in predicciones.items()}
hombre = (prueba.sex == 'male').to_numpy()
parcial = ~np.asarray(mascara['test']).all(axis=(1, 2))  # ECG de EchoNext con tramos vacíos sin marcar (~1 %)

resultados = {
    'semilla': SEMILLA, 'etiqueta': ETIQUETA, 'gpu': GPU, 'version_paquete': VERSION_PAQUETE,
    'formato_unico': formato, 'alineacion_con_metadata': alineacion,
    'red': {'parametros': PARAMETROS, 'epocas_max': EPOCAS, 'paciencia': PACIENCIA, 'lote': LOTE, 'tasa': TASA,
            'historial': historiales,
            'mejor_epoca': {m: max(h, key=lambda r: r['auc_val'])['epoca'] for m, h in historiales.items()}},
    'test': {
        n: {**metricas(y_test, p, umbrales[n]), 'auc_ic95': intervalo(muestras[n]), 'umbral_youden_val': round(umbrales[n], 4),
            'auprc': round(float(average_precision_score(y_test, p)), 4), 'brier': round(float(brier_score_loss(y_test, p)), 4)}
        for n, p in puntajes.items()
    },
    'diferencias_auc': {
        f'{a}_menos_{b}': {'media': round(float((muestras[a] - muestras[b]).mean()), 4), 'ic95': intervalo(muestras[a] - muestras[b])}
        for a, b in [('red_10s', 'red_formato_pdf'), ('red_formato_pdf', 'boosting_tabular'), ('red_10s', 'boosting_tabular')]
    },
    'por_sexo': {s: {n: metricas(y_test[m], p[m], umbrales[n]) for n, p in puntajes.items()} for s, m in [('female', ~hombre), ('male', hombre)]},
    'por_edad': {t: {n: auc_seguro(y_test[m], p[m]) for n, p in puntajes.items()}
                 for t in sorted(prueba.tramo_edad.unique()) for m in [(prueba.tramo_edad == t).to_numpy()]},
    'ecg_parciales_test': {'cantidad': int(parcial.sum()),
                           'auc_sin_parciales': {n: auc_seguro(y_test[~parcial], p[~parcial]) for n, p in puntajes.items()}},
}
print(json.dumps({k: resultados[k] for k in ['test', 'diferencias_auc', 'ecg_parciales_test']}, indent=1, ensure_ascii=False))

In [ ]:
# Celda 7 — Gráficos (curvas agregadas, nunca un ECG individual) y guardado
import matplotlib.pyplot as plt

colores = {'red_10s': '#534AB7', 'red_formato_pdf': '#D85A30', 'boosting_tabular': '#888780'}
fig, ejes = plt.subplots(1, 2, figsize=(12, 5))
for n, p in puntajes.items():
    fpr, tpr, _ = roc_curve(y_test, p)
    ejes[0].plot(fpr, tpr, color=colores[n], lw=2, label=f"{n} (AUC {resultados['test'][n]['auc']:.3f})")
ejes[0].plot([0, 1], [0, 1], color='0.7', ls='--', lw=1)
ejes[0].set(title='Curva ROC en test', xlabel='1 − especificidad', ylabel='Sensibilidad')
ejes[0].legend(fontsize=9)
for modo, h in historiales.items():
    ejes[1].plot([r['epoca'] for r in h], [r['auc_val'] for r in h], color=colores[NOMBRES[modo]], lw=2, marker='o', label=NOMBRES[modo])
ejes[1].axhline(roc_auc_score(y_val, predicciones['boosting_tabular']['val']), color=colores['boosting_tabular'], ls='--', label='boosting_tabular (val)')
ejes[1].set(title='AUC en val por época', xlabel='Época', ylabel='AUC')
ejes[1].legend(fontsize=9)
fig.suptitle('E-007 — Red desde cero: 10 s frente al formato del PDF (EchoNext, agregados)')
fig.tight_layout()
ruta_figura = CARPETA / 'resultados' / 'E-007_red_formato_pdf.png'
fig.savefig(ruta_figura, dpi=130)
plt.show()

ruta_resultados = CARPETA / 'resultados' / 'E-007_red_formato_pdf.json'
ruta_resultados.write_text(json.dumps(resultados, indent=2, ensure_ascii=False))
print('Guardado:', ruta_resultados.name, ruta_figura.name)
print(json.dumps({k: resultados[k] for k in ['por_sexo', 'por_edad']}, indent=1, ensure_ascii=False))